In [1]:
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpad\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @C:\Users\miao\Documents\Database
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc1\userspace\miao\cluster"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc1\userspace\miao\cluster"


In [4]:
static var myBatch = ExecutionQueues[2];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Droplet-EE");

In [5]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE");

Project name is set to 'Droplet-EE'.
Default Execution queue is chosen for the database.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

In [7]:
//static var myDb = OpenOrCreateDatabase(@"C:\Databases\BoSSS_DB");
//BoSSSshell.WorkflowMgm.Init("HeatedSquareCavity");

## Create grid

In [8]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xSize = 6;
        double yTop = (3.5 - 0.00625);
        double yBottom = (- 2.5 - 0.00625);
        int kelem = 36 * 7 / 2;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 2 + 1);
                var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: false);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                //grd.EdgeTagNames.Add(3, "wall_left");
                //grd.EdgeTagNames.Add(4, "wall_right");
                grd.EdgeTagNames.Add(3, "FreeSlip_left");
                grd.EdgeTagNames.Add(4, "FreeSlip_right");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                        et = 3;
                    if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                        et = 4;

                    return et;
                });

                return grd;
     }
 
 }

In [9]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double VelX(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double VelY(double[] X) {");
           stw.WriteLine("    return 0.046 / 1.778;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return (((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2()).Sqrt() - 1.778);");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] - 0.001);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_VelX() {
        return new Formula("BoundaryValues.VelX", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_VelY(){
        return new Formula("BoundaryValues.VelY", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [10]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;

            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            int D = 2;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            C.DbPath = @"\\dc1\userspace\miao\cluster\Droplet-EE";
            C.ProjectName = "Droplet";
            C.SessionName = "Droplet-check-BDF2-FreeSlip-NoR-FinerMesh";
            C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            // Physical Parameters
            // ===================
            #region physics

            double scale = 1e-4;
            C.PhysicalParameters.rho_A = 1 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false;

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale
                //Viscosity = 0 0.05e-4 * scale,
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            #endregion

            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_VelY());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_VelX());
            C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
            //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());



            // boundary conditions
            // ===================
            #region BC


            C.AddBoundaryValue("wall_lower");
            C.AddBoundaryValue("pressure_outlet_upper");
            //C.AddBoundaryValue("wall_left");
            //C.AddBoundaryValue("wall_right");
            C.AddBoundaryValue("FreeSlip_left");
            C.AddBoundaryValue("FreeSlip_right");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 160;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = false;
            //C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
            //C.AMR_startUpSweeps = 4;



            #endregion


            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 5e-7;
            double dt = 5e-3;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 10000;
            C.saveperiod = 1;

            #endregion

            return C;
        }

## Send and run jobs

In [11]:
    var C_CTRLFILE = Aland(2, 0);//Create control file.

Grid Edge Tags changed.


In [12]:
    var JobLocal = C_CTRLFILE.CreateJob();

In [13]:
JobLocal.Status

PreActivation

In [14]:
    JobLocal.NumberOfMPIProcs = 2;
    JobLocal.Activate();
    JobLocal.ShowOutput();
    //BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

Deploying job Droplet-check-BDF2-FreeSlip-NoR-FinerMesh ... 
Grid is not in database yet...
Grid successfully saved: 20cc637c-18c5-4c6e-94c3-4399c69c72fc
Deploying executables and additional files ...
Deployment directory: \\dc1\userspace\miao\cluster\Droplet-EE-ZwoLevelSetSolver2023Oct27_210317.006186
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [8]:
//JobLocal.Stdout

In [8]:
wmg.Sessions

#0: Droplet-EE	Droplet-check-BDF2-FreeSlip-NoR-FinerMesh*	10/27/2023 9:03:52 PM	45f7e49c...
#1: Droplet-EE	Droplet-check-BDF2-FreeSlip-NoR-FinerMesh*	10/27/2023 7:23:10 AM	20402df9...
#2: Droplet-EE	Droplet-check-BDF2-FreeSlip-NoR-FinerMesh	10/27/2023 7:12:09 AM	4441ff8e...
#3: Droplet-EE	Droplet-check-BDF2-FreeSlip-NoR-FinerMesh	10/27/2023 7:00:43 AM	47de4e5b...
#4: Droplet-EE	Droplet-check-BDF2-FreeSlip-NoR-FinerMesh	10/27/2023 6:52:20 AM	f073cd35...
#5: Droplet-EE	Droplet-check-BDF2-FreeSlip-NoR	10/26/2023 8:49:42 PM	3421bce3...
#6: Droplet-EE	Droplet-check-BDF2-FreeSlip-Agg0.1*	10/25/2023 11:41:11 AM	29837e6d...
#7: Droplet-EE	Droplet-check-ImplicitEuler-FreeSlip-Agg0.1*	10/25/2023 11:33:31 AM	b409332e...
#8: Droplet-EE	Droplet-check-ImplicitEuler-FreeSlip*	10/25/2023 11:28:40 AM	1ce66971...
#9: Droplet-EE	Droplet-check-ImplicitEuler*	10/24/2023 2:32:23 PM	8f22ce6b...
#10: Droplet-EE	Droplet-check-BDF3*	10/24/2023 2:29:15 PM	fb77b31a...
#11: Droplet-EE	Droplet-check*	10/24/2023 10:

In [13]:
wmg.Sessions[0].Timesteps.Count

1065

In [14]:
var outPath = wmg.Sessions[0].Timesteps.Every(10).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE__Droplet-check-BDF2-FreeSlip-NoR-FinerMesh__45f7e49c-5d17-4129-b9a1-aed4c0717bc6


## Post processing

In [19]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [20]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");